Step 1:

First of all, we need to load the given dataset MLoGPU_data3_train.csv into Google Colab. We have checked the shape, preview the first rows, and then make separation of the features and labels. This assists us to ensure that the data is well suited for our further processing.


In [10]:
import numpy as np

# Here we have loaded the CSV file. This file has total 8 columns(7 features + 1 class label)
data = np.loadtxt("MLoGPU_data3_train.csv", delimiter=",")

# The we have checked the shape to confirm the number of samples and features
print("Data structure:", data.shape)

# We have depicted the first 10 rows to get to know about the structure
print("First 10 rows:")
print(data[:10])

# Then we have separated features (X) and labels (y)
X = data[:, :-1] # First 7 colums for features
y = data[:, -1] # Last colum for class label

print("Feature matrix structure:", X.shape)
print("Label vector structure:", y.shape)

# We have also examined the unique class labels
print("All unique classes:", np.unique(y))

Data structure: (4000, 8)
First 10 rows:
[[ 0.147035  1.264512 -0.474615 -0.543303 -1.091289  0.590143 -1.599582
   4.      ]
 [ 0.559923  1.935253 -2.557251 -0.743111 -0.69648  -0.085936 -1.899705
   6.      ]
 [ 0.766367 -0.496182  0.373866  0.170298 -0.54463  -0.592996  0.634732
   4.      ]
 [-1.642146  0.426087  0.913809  1.397691  1.338306 -0.818356  0.367963
   3.      ]
 [-0.334668  0.090717  1.608021  0.627002  2.644213 -0.367636  1.234998
   4.      ]
 [-2.192664  0.593772 -0.937423 -0.400583  0.336098 -1.325415 -1.116034
   2.      ]
 [-0.265853  1.599882 -0.320346 -0.628935 -0.119451 -0.311296 -1.479525
   5.      ]
 [-0.128224  0.677615 -0.783154 -0.714567 -0.969809 -0.423976 -1.459509
   4.      ]
 [ 1.179255  0.593772  0.682405 -0.543303 -0.240931 -0.254956 -0.132262
   5.      ]
 [ 1.179255 -1.083079 -0.628884 -0.286407 -0.42315   1.266222  0.301256
   4.      ]]
Feature matrix structure: (4000, 7)
Label vector structure: (4000,)
All unique classes: [1. 2. 3. 4. 5. 6. 7

Step 2:

In the 2nd step, we have prepared the dataset since we have to use it efficiently for both CPU and GPU implementations of kNN. We have standardized(convert every feature in a general scale) the feature values because kNN makes predictions by distance-based measurment. Feature values without scaling can badly effect the final predictions. That's why scaling is a must for kNN. We also split the data into training and validation sets so that after the training phase, we can evaluate the classifier properly. We have tried to keep the preprocessing simple and follows standard practices in machine learning.



In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Here we have split the data into training and validation sets. For training we have used 80% of data and for validation we have used 20%
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print("Training set structure:", X_train.shape)
print("Validation set structure:", X_val.shape)

# The we have done the standardization of the features so that all dimensions have similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

print("Feature scaling completed.")

Training set structure: (3200, 7)
Validation set structure: (800, 7)
Feature scaling completed.


Step 3:

Now, in this step, we have simply implemented a CPU version of the k‑nearest‑neighbour classifier. This helps us to verify correctness before moving to the GPU implementation. We have computed distances using NumPy, then we have selected the k nearest neighbours, and predicted the class by majority voting. We have also measured the execution time so that we can make a comparison with the GPU version.


In [15]:
import numpy as np
import time

# Firstly, we have defined a function to compute Euclidean distances on the CPU
def compute_distances_cpu(X_train, X_test):
    # We have used broadcasting(that means, NumPy will automatically adjust arrays so their shapes can be matched for math operations) to compute all pairwise distances.
    pair_dists = np.sqrt(
        np.sum((X_train[None, :, :] - X_test[:, None, :])**2, axis=2)
    )
    return pair_dists


# Then we have defined a simple kNN classifier on CPU
def knn_cpu(X_train, y_train, X_test, k=5):
    # Then we have computed distances between testing and training samples
    pair_dists = compute_distances_cpu(X_train, X_test)

    # Now we need to find the indices of the k nearest neighbours
    knn_indx = np.argsort(pair_dists, axis = 1)[:, :k]

    # Here we have gathered the labels of the neighbours
    knn_labels = y_train[knn_indx]

    # Finally we just need to make prediction by majority vote
    preds = np.array([np.bincount(row.astype(int)).argmax() for row in knn_labels])
    return preds

# Now we need to keep the current time before running the kNN for calculating the execution time later
init_time = time.time()

# Here we have run the kNN classifier and store the predictions
y_pred_cpu = knn_cpu(X_train, y_train, X_val, k = 5)

# Finally we calculate the execution time here
cpu_exec_time = time.time() - init_time

print("CPU kNN prediction completed. CPU execution time is (sec):", cpu_exec_time)

# After comparing the predicted labels with the true labels, we have computed the accuracy on the validation set
acc_cpu = np.mean(y_pred_cpu == y_val)
print("CPU validation accuracy is:", acc_cpu)

CPU kNN prediction completed. CPU execution time is (sec): 0.2257094383239746
CPU validation accuracy is: 0.53


Step 4

Here we have just verified the CuPy version and type of GPU device


In [16]:
import cupy as cp

print("CuPy version:", cp.__version__)
print("GPU device:", cp.cuda.runtime.getDeviceProperties(0)["name"])

CuPy version: 14.0.1
GPU device: b'Tesla T4'


Step 5:

Now we have the outcomes from cpu. In this step, we will build the GPU version of kNN. We are asked to compute the distances using CUDA kernel also and this kernel should be written with CuPy's. We have tried to keep the kernel simple and then measure the Euclidean distances between each validation sample and all training samples.

In [17]:
import cupy as cp

# First of all, we need to transfer training and validation data to the GPU using "cp.asarray"
X_train_gpu = cp.asarray(X_train)
X_val_gpu = cp.asarray(X_val)

# We need an empty matrix to store distances in float64
pair_dists_gpu = cp.zeros((X_val_gpu.shape[0], X_train_gpu.shape[0]), dtype = cp.float64)

# Here we have initialized a CUDA kernel and named the GPU function "compute_distances"

kernel_code = r'''
extern "C" __global__
void compute_distances(const double* X_train, const double* X_test,
                       double* dists, int N_train, int N_test, int dim)
{

  // Since each GPU thread handles one pair of train and test sample, we have defined the train and test indices here

    int i = blockIdx.x * blockDim.x + threadIdx.x;  // test index
    int j = blockIdx.y * blockDim.y + threadIdx.y;  // train index

    if (i < N_test && j < N_train)
      {

        double sum = 0.0;

        // Similar like CPU
        for (int k = 0; k < dim; k++) {
            double a = X_test[i * dim + k];
            double b = X_train[j * dim + k];
            double diff = a - b;
            sum += diff * diff;
      }

        dists[i * N_train + j] = sqrt(sum);
    }
}
'''

# Here we have compiled the CUDA code into GPU executable kernel
distance_cuda_kernel = cp.RawKernel(kernel_code, "compute_distances")

# We have blocks of 16x16(GPU threads) sizes
block = (16, 16)

# Then we have computed how many blocks we need to store all the test and train pairs
grid = ((X_val_gpu.shape[0] + block[0] - 1) // block[0],
        (X_train_gpu.shape[0] + block[1] - 1) // block[1])


# Now we are launching the kernel
distance_cuda_kernel(
    grid, block,
    (
        X_train_gpu.ravel(),
        X_val_gpu.ravel(),
        pair_dists_gpu,
        X_train_gpu.shape[0],
        X_val_gpu.shape[0],
        X_train_gpu.shape[1]
    )
)

print("GPU distance matrix for first three rows:")
print(pair_dists_gpu[:3, :3])

GPU distance matrix for first three rows:
[[1.57023893 3.70921406 2.69827245]
 [4.44028836 5.03549829 2.62191741]
 [2.77350553 4.44128341 1.69643521]]


Step 6:

Next, we will do the GPU version of the k‑nearest‑neighbour classifier.
We have already measured the full distance matrix on the GPU with double precision custom CUDA kernel.
Now we will at first select the k nearest neighbors for each validation sample with GPU operations. Secondly, we will arrange the labels from training ser. Then for determining the predicted class, we will apply the majority voting mechanism. Lastly, we will compare the accuracy and runtime with CPU outcomes


In [18]:
k = 5

# Firstly, we need to save the current time before the GPU operations start
init_time = time.time()

# Here we have found the k smallest distances means k closest training samples for each test samples.
# Now we are sorting the indices
knn_indx_gpu = cp.argsort(pair_dists_gpu, axis = 1)[:, :k]

# Here we have chose the labels of k nearest neighbors for each test point
y_train_gpu = cp.asarray(y_train)
knn_labels_gpu = y_train_gpu[knn_indx_gpu]

# Then finally we have done the majority voting here. We have used "cp.bincount" because it is safe and works perfectly on GPU
preds_gpu = cp.array([
    cp.bincount(row.astype(cp.int32)).argmax()
    for row in knn_labels_gpu
])

# Here we have calculated the GPU time
gpu_exec_time = time.time() - init_time

# Then we have simple shifted the prediction to CPU so that we can use them in python
y_pred_gpu = cp.asnumpy(preds_gpu)
print("GPU kNN prediction completed. GPU execution time is (sec):", gpu_exec_time)


GPU kNN prediction completed. GPU execution time is (sec): 0.3739125728607178


Step 7:

In this final step, we have just compared the accuracy and execution time between CPU and GPU. Since we have a small dataset, we have got a less execution time for CPU than GPU. But for larger dataset GPU can perform better then CPU and in that case, parallelism can be fully utilized. We have also measured the speed variation of CPU over GPU.


In [19]:
acc_gpu = np.mean(y_pred_gpu == y_val)

print("CPU accuracy:", acc_cpu)
print("GPU accuracy:", acc_gpu)
print("CPU time in seconds:", cpu_exec_time)
print("GPU time in seconds):", gpu_exec_time)
print("GPU vs CPU(Faster Ratio):", cpu_exec_time / gpu_exec_time)

CPU accuracy: 0.53
GPU accuracy: 0.53
CPU time in seconds: 0.2257094383239746
GPU time in seconds): 0.3739125728607178
GPU vs CPU(Faster Ratio): 0.6036422808602674
